In [ ]:
import pandas as pd
import numpy as np

In [ ]:
file_path = 'retail_data.csv'
df = pd.read_csv(file_path, sep=',')

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 78 columns):
 #   Column                     Non-Null Count    Dtype  
---  ------                     --------------    -----  
 0   customer_id                1000000 non-null  int64  
 1   age                        1000000 non-null  int64  
 2   gender                     1000000 non-null  object 
 3   income_bracket             1000000 non-null  object 
 4   loyalty_program            1000000 non-null  object 
 5   membership_years           1000000 non-null  int64  
 6   churned                    1000000 non-null  object 
 7   marital_status             1000000 non-null  object 
 8   number_of_children         1000000 non-null  int64  
 9   education_level            1000000 non-null  object 
 10  occupation                 1000000 non-null  object 
 11  transaction_id             1000000 non-null  int64  
 12  transaction_date           1000000 non-null  object 
 13  product_id   

In [ ]:
import pandas as pd
import numpy as np

# 1. Доли операций (Фильтрация через рандом)
# Создаем временную колонку с рандомом
df['rnd'] = np.random.rand(len(df))

# Условия для фильтрации
masks = (
    ((df["payment_method"] == "Credit Card") & (df["rnd"] < 0.85)) |
    ((df["payment_method"] == "Debit Card") & (df["rnd"] < 0.75)) |
    ((df["payment_method"] == "Cash") & (df["rnd"] < 0.55)) |
    ((df["payment_method"] == "Mobile Payment") & (df["rnd"] < 1.0))
)

df = df[masks].drop(columns=['rnd'])

# 2. Количество товаров (Генерация через нормальное распределение)
# np.random.randn генерирует стандартное нормальное распределение (среднее 0, откл 1)
# Аналог spark_round — это метод .round()
conditions = [
    (df["payment_method"] == "Cash"),
    (df["payment_method"] == "Debit Card"),
    (df["payment_method"] == "Credit Card")
]

choices = [
    (np.random.randn(len(df)) * 0.5 + 1.5).round(),
    (np.random.randn(len(df)) * 0.8 + 2.5).round(),
    (np.random.randn(len(df)) * 1.2 + 3.5).round()
]

# np.select — это прямой аналог цепочки when().otherwise()
df["quantity"] = np.select(conditions, choices, default=(np.random.randn(len(df)) * 1.8 + 4.5).round())

# Ограничение снизу (минимум 1 товар)
df["quantity"] = df["quantity"].clip(lower=1)

# 3. Усиление ценового сегмента
price_conditions = [
    (df["payment_method"] == "Cash"),
    (df["payment_method"] == "Debit Card"),
    (df["payment_method"] == "Credit Card")
]

price_multipliers = [
    df["unit_price"] * 0.8,
    df["unit_price"] * 1.0,
    df["unit_price"] * 1.2
]

df["unit_price"] = np.select(price_conditions, price_multipliers, default=df["unit_price"] * 1.35)

# 4. Реалистичная модель скидки
# least(col + 0.1, 0.5) в Pandas заменяется на .clip(upper=0.5)
mask_discount = (df["payment_method"] == "Mobile Payment") & (df["quantity"] * df["unit_price"] > 300)

df.loc[mask_discount, "discount_applied"] = (df["discount_applied"] + 0.1).clip(upper=0.5)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 787114 entries, 0 to 999999
Data columns (total 78 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   customer_id                787114 non-null  int64  
 1   age                        787114 non-null  int64  
 2   gender                     787114 non-null  object 
 3   income_bracket             787114 non-null  object 
 4   loyalty_program            787114 non-null  object 
 5   membership_years           787114 non-null  int64  
 6   churned                    787114 non-null  object 
 7   marital_status             787114 non-null  object 
 8   number_of_children         787114 non-null  int64  
 9   education_level            787114 non-null  object 
 10  occupation                 787114 non-null  object 
 11  transaction_id             787114 non-null  int64  
 12  transaction_date           787114 non-null  object 
 13  product_id                 787114 

In [ ]:
df.to_csv('my_data.csv', index=False)